Project #1 - environment configuration

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import dcc, html, Input, Output
from pathlib import Path

Project #1 - importing data

Load the OWID-style CSV from the project root. Cleaning is applied per visualization where needed.


In [ ]:
DATA_PATH = Path("project_1_python.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. "
        "Place project_1_python.csv in the project root."
    )

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.rename(columns={"location": "country"})

df

In [ ]:
df.info()

Project #1 - first visualization

In [ ]:
df_countries = df[["iso_code","continent","country","population","life_expectancy"]].drop_duplicates()
df_countries = df_countries.sort_values(by="population",ascending=False)

ten_top_pop_countries = df_countries.head(10)

ten_top_pop_countries


In [ ]:
graph = sns.catplot(
    data=ten_top_pop_countries,
    x=ten_top_pop_countries["population"]/1e6,
    y=ten_top_pop_countries["country"],
    kind="bar")

sns.set_style("whitegrid")
plt.xlabel("population(millions)")
graph.set_xticklabels(rotation=45, ha="right")

Project #1 - population vs. life expectancy

In [ ]:
fig = px.scatter(
    data_frame=df_countries,
    x="population",
    y="life_expectancy",
    title="Population vs life expectancy",
    log_x=True,
    color="continent",
    hover_name="country",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.show()

Project #1 - number of diagnosed cases

In [ ]:
df_cz_de = df[df["country"].isin(["Czechia", "Germany"])].copy()
df_cz_de

In [ ]:
df_cz_de["new_cases_per_1000_people"] = (
    df_cz_de["new_cases"] / df_cz_de["population"] / 1000
)

fig = px.line(
    data_frame=df_cz_de,
    x="date",
    y="new_cases_per_1000_people",
    color="country",
    color_discrete_sequence=px.colors.qualitative.Antique,
    title="Covid cases in Czechia vs Germany per thousand people over time",
)
fig.update_yaxes(exponentformat="none")
fig.show()

In [ ]:
# now I am going to rescale it to population of both countries:

df_cz_de["new_cases_per_1000_people"] = df_cz_de["new_cases"]/df_cz_de["population"]/1000

fig = px.line(
    data_frame=df_cz_de,
    x="date",
    y="new_cases_per_1000_people",
    color="country",
    color_discrete_sequence=px.colors.qualitative.Antique,
    title="Covid cases in Czechia vs Germany per thousand people over time"
)

# this line is from Claude, it fixes y axis values to something not containing Greek letters:
fig.update_yaxes(exponentformat="none") 

fig.show()

# Latest reported row per country; marker size = cases / population
df_countries2 = (
    df.sort_values("date")
    .drop_duplicates(subset="country", keep="last")
    .dropna(subset=["total_cases", "population", "iso_code"])
    .copy()
)
df_countries2["cases_to_population"] = (
    df_countries2["total_cases"] / df_countries2["population"]
)

fig = px.scatter_geo(
    data_frame=df_countries2,
    locations="iso_code",
    locationmode="ISO-3",
    color="continent",
    size="cases_to_population",
    hover_name="country",
    hover_data=["total_cases", "date"],
    title="Cases to population (latest report per country)",
)
fig.update_layout(template="plotly_dark", height=800)
fig.show()

In [ ]:

"""
I am not sure how to understand
the task "2. Make the size of the marker dependent on the ratio of cases to population in the country."
There are multiple values of total cases for each country.
Anyway, I've decided to pick the highest value, presumably this is total cases for all time 
(to publication of dataset, anyway).
"""
df_countries2 = df.sort_values("total_cases", ascending=False).drop_duplicates(subset="country", keep="first")

df_countries2["cases_to_population"] = df_countries2["total_cases"]/df_countries2["population"]

fig = px.scatter_geo(
    data_frame=df_countries2,
    locations="iso_code",
    locationmode="ISO-3",      
    color="continent",
    size="cases_to_population",             
    hover_name="country",       
    hover_data="total_cases",
    title="Cases to population"
)

fig.update_layout(template="plotly_dark",height=800)

fig.show()


app1 = dash.Dash(__name__)
font_styles = {"fontFamily": "verdana", "color": "#444"}

app1.layout = html.Div(
    [
        html.H3(id="title", style=font_styles),
        dcc.Dropdown(
            id="country-picker",
            clearable=False,
            options=[
                {"label": country, "value": country}
                for country in sorted(df["country"].unique())
            ],
            value="Czechia",
            style=font_styles,
        ),
        html.Div(
            [
                dcc.Graph(id="graph_cases", style={"width": "50%"}),
                dcc.Graph(id="graph_deaths", style={"width": "50%"}),
            ],
            style={"display": "flex"},
        ),
    ]
)


@app1.callback(
    Output("title", "children"),
    Output("graph_cases", "figure"),
    Output("graph_deaths", "figure"),
    Input("country-picker", "value"),
)
def update_country_dashboard(country):
    title = f"Cumulative number of positive cases and deaths in {country}"
    selected_country = df[df["country"] == country]
    cases = px.line(selected_country, x="date", y="total_cases", title="Total cases")
    cases.update_layout(xaxis_title="Date", yaxis_title="Cases")
    deaths = px.line(selected_country, x="date", y="total_deaths", title="Total deaths")
    deaths.update_layout(xaxis_title="Date", yaxis_title="Deaths")
    return title, cases, deaths


if __name__ == "__main__":
    app1.run(debug=False, host="127.0.0.1", port=8050)

In [ ]:
app = dash.Dash()

font_styles={"fontFamily": "verdana", "color": "#444"}

app.layout = html.Div(
    [html.H3(id="title", style=font_styles),
     
dcc.Dropdown(
        id="country-picker",
        clearable=False,
        options=[{"label": country, "value": country} 
                 for country in df["country"].unique()],
        value = "Czechia",
        style=font_styles
        ),
html.Div([
dcc.Graph(id="graph_cases", style={'width': '50%'}),
dcc.Graph(id="graph_deaths", style={'width': '50%'})
], style = {"display":"flex"})

])

@app.callback(
    Output("title", "children"),
    Output("graph_cases", "figure"),
    Output("graph_deaths", "figure"),
    Input("country-picker","value")
)
def update_dashboard(country):
    title = f"Cumulative number of positive cases and deaths in {country}"
    selected_country = df[df["country"] == country]
    cases = px.line(selected_country, x="date", y="total_cases")
    deaths = px.line(selected_country, x="date", y="total_deaths")
    return title, cases, deaths

if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=8050)

# One row per country: each country's latest reported values
df_latest = (
    df.sort_values("date")
    .drop_duplicates(subset="country", keep="last")
    .reset_index(drop=True)
)

print(f"Countries in snapshot: {len(df_latest)}")
print(f"Reporting dates range from {df_latest['date'].min().date()} to {df_latest['date'].max().date()}")
df_latest.head()

In [ ]:
df_latest

In [ ]:
df_latest.info()

# total_tests is often missing on the latest row; dashboard 2 drops nulls per metric.

In [ ]:
METRIC_OPTIONS = {
    "total number of positive cases": "total_cases",
    "total number of deaths": "total_deaths",
    "total number of tests": "total_tests",
    "total number of vaccinations": "total_vaccinations",
    "total number of vaccinated people": "people_vaccinated",
}

DEFAULT_CONTINENT = "Europe"
DEFAULT_METRIC = "total number of positive cases"

app2 = dash.Dash(__name__)
font_styles = {"fontFamily": "verdana", "color": "#444"}

app2.layout = html.Div(
    [
        html.H2(id="title"),
        html.Div(
            [
                html.Div(
                    [
                        html.H3("Continent:"),
                        dcc.Dropdown(
                            id="continent-picker",
                            options=[
                                {"label": continent, "value": continent}
                                for continent in sorted(df_latest["continent"].dropna().unique())
                            ],
                            value=DEFAULT_CONTINENT,
                            clearable=False,
                        ),
                    ],
                    style={"margin": "10px"},
                ),
                html.Div(
                    [
                        html.H3("Metric:"),
                        dcc.Dropdown(
                            id="metric-picker",
                            options=list(METRIC_OPTIONS.keys()),
                            value=DEFAULT_METRIC,
                            clearable=False,
                        ),
                    ],
                    style={"margin": "10px"},
                ),
            ],
            style={"display": "flex"},
        ),
        dcc.Graph(id="map"),
    ],
    style=font_styles,
)


@app2.callback(
    Output("title", "children"),
    Output("map", "figure"),
    Input("continent-picker", "value"),
    Input("metric-picker", "value"),
)
def update_continent_map(continent, metric):
    if continent is None or metric is None:
        return "Select a continent and a metric", go.Figure()

    column = METRIC_OPTIONS[metric]
    continent_data = df_latest[df_latest["continent"] == continent]
    before_count = len(continent_data)
    continent_data = continent_data.dropna(subset=[column])
    after_count = len(continent_data)

    if continent_data.empty:
        empty = go.Figure()
        empty.update_layout(
            title=f"No data for '{metric}' in {continent}",
            template="plotly_dark",
        )
        return (
            f"COVID-19 - {metric} in {continent} (no countries with data)",
            empty,
        )

    excluded = before_count - after_count
    title = f"COVID-19 - {metric} in {continent}"
    if excluded:
        title += f" ({excluded} countries hidden due to missing data)"

    geo_fig = px.scatter_geo(
        data_frame=continent_data,
        locations="iso_code",
        locationmode="ISO-3",
        size=column,
        hover_name="country",
        title=title,
    )
    geo_fig.update_layout(template="plotly_dark", height=800)
    return title, geo_fig


if __name__ == "__main__":
    app2.run(debug=False, host="127.0.0.1", port=8051)

In [ ]:
app = dash.Dash()

font_styles={"fontFamily": "verdana", "color": "#444"}

app.layout = html.Div([
    
    html.H2(id="title"),

    html.Div([

    html.Div([
    html.H3("Continent:"),
    dcc.Dropdown(
        id="continent-picker",
        options=[{"label": continent, "value": continent} 
                 for continent in df["continent"].unique()],
        clearable=False
    )], style= {"margin": "10px"},
    id="continent-dropdown"),
    
    html.Div([
    html.H3("Metric:"),
    dcc.Dropdown(
        id="metric-picker",
        options = [
            "total number of positive cases",
            "total number of deaths",
            "total number of tests",
            "total number of vaccinations",
            "total number of vaccinated people"
        ]
    )], style= {"margin": "10px"},
    id="metric-dropdown"
    )],

    style = {"display":"flex"},
    id="dropdowns"),

    html.Div(
    dcc.Graph(id="map"),
    id="maps")

],style=font_styles,
id="container"
    
)

@app.callback(
    Output("title", "children"),
    Output("map", "figure"),
    Input("continent-picker", "value"),
    Input("metric-picker","value")
    
)
def update_dashboard(continent, metric):

    if continent is None or metric is None:
        return "Select a continent and a metric", go.Figure()
    
    title = f"COVID-19 - {metric} in {continent}"
    
    selected_continent = df_last_day[df_last_day["continent"] == continent]
    
    match metric:
        case "total number of positive cases":
            selected_metric = "total_cases"
        case "total number of deaths":
            selected_metric = "total_deaths"
        case "total number of tests":
            selected_metric = "total_tests"
        case "total number of vaccinations":
            selected_metric = "total_vaccinations"
        case "total number of vaccinated people":
            selected_metric = "people_vaccinated"

    selected_continent = selected_continent.dropna(subset=[selected_metric])

    map = px.scatter_geo(
        data_frame=selected_continent,
        locations="iso_code",
        locationmode="ISO-3",      
        size=selected_metric,             
        hover_name="country"       
        )
    map.update_layout(template="plotly_dark",height=800)
    return title, map

if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=8051)


# Top-N countries by total vaccinations (latest report per country)
df_latest_sorted_by_vax = df_latest.sort_values(
    by="total_vaccinations",
    ascending=False,
    na_position="last",
)
top_20_in_vaccinations = (
    df_latest_sorted_by_vax.dropna(subset=["total_vaccinations"]).head(20)
)
top_20_in_vaccinations

In [ ]:
df_latest_vax = df_latest.dropna(subset=["people_vaccinated", "population"]).copy()
df_latest_vax["vaccinated_share"] = (
    df_latest_vax["people_vaccinated"] / df_latest_vax["population"]
)

top_20_in_ratio = (
    df_latest_vax.sort_values(by="vaccinated_share", ascending=False)
    .head(20)
)
top_20_in_ratio

In [ ]:
app3 = dash.Dash(__name__)
font_styles = {"fontFamily": "verdana", "color": "#444"}

app3.layout = html.Div(
    [
        html.Div(
            [
                html.H1("Covid statistics"),
                html.H3("Choose number of countries shown in the charts:"),
                dcc.Dropdown(
                    id="number-picker",
                    options=[
                        {"label": str(n), "value": n} for n in [5, 10, 15, 20]
                    ],
                    clearable=False,
                    value=5,
                ),
            ]
        ),
        html.Div(
            [
                dcc.Graph(id="graph_vaccinations", style={"width": "50%"}),
                dcc.Graph(id="graph_vax_ratio", style={"width": "50%"}),
            ],
            style={"display": "flex"},
        ),
    ],
    style=font_styles,
)


@app3.callback(
    Output("graph_vaccinations", "figure"),
    Output("graph_vax_ratio", "figure"),
    Input("number-picker", "value"),
)
def update_vaccination_dashboard(no_of_countries):
    df_vax = top_20_in_vaccinations.head(no_of_countries)
    df_ratio = top_20_in_ratio.head(no_of_countries)

    vaccinations = px.bar(
        data_frame=df_vax,
        x="country",
        y="total_vaccinations",
        title=f"Top {no_of_countries} countries by total vaccinations",
        labels={"total_vaccinations": "Total vaccinations"},
    )

    ratios = px.bar(
        data_frame=df_ratio,
        x="country",
        y="vaccinated_share",
        title=f"Top {no_of_countries} countries by vaccination coverage",
        labels={"vaccinated_share": "Share of population vaccinated"},
    )

    vaccinations.update_layout(height=600)
    ratios.update_layout(height=600)

    return vaccinations, ratios


if __name__ == "__main__":
    app3.run(debug=False, host="127.0.0.1", port=8052)

In [ ]:
app = dash.Dash()

font_styles={"fontFamily": "verdana", "color": "#444"}

app.layout = html.Div([

    html.Div([
        html.H1("Covid statistics"),
        html.H3("Choose number of countries shown in the charts:"),
        html.Div(dcc.Dropdown(
            id="number-picker",
            options=[5, 10, 15, 20],
            clearable=False,
            value=5
        ), id="dropdown-container")
    ]),

    html.Div([
        dcc.Graph(id="graph_cases", style={'width': '50%'}),
        dcc.Graph(id="graph_vax_ratio", style={'width': '50%'})
    ], style={"display": "flex"},
       id="graphs-container"
    )

], style=font_styles,
   id="main-container")

@app.callback(
    Output("graph_cases","figure"),
    Output("graph_vax_ratio","figure"),
    Input("number-picker", "value")
)
def update_dashboard(no_of_countries):
    df_cases = top_20_in_cases.head(no_of_countries)
    df_ratio = top_20_in_ratio.head(no_of_countries)

    cases = px.bar(
        data_frame=df_cases,
        x="country",
        y="total_cases",
        labels = {"total_cases": "<b>Total covid cases</b>"}
    )

    ratios = px.bar(
        data_frame=df_ratio,
        x="country",
        y="ratio",
        labels = {"ratio": "<b>Ratio of vaccinations to population</b>"}
    )

    cases,ratios.update_layout(height=600)

    return cases, ratios

if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=8052)